# NLP Lab Assignment 9: MEMM and CRF Hindi POS Tagging

This notebook implements the Assignment 9 requirements on the Hindi POS dataset used in Assignment 7:

1. A Maximum Entropy Markov Model (MEMM).
2. A Conditional Random Field (CRF).
3. A comparison with the Naive Bayes and HMM results from Assignment 7.

The feature design follows the assignment: prefixes and suffixes of lengths 1--4, previous/current/next words, and a binary word-length feature. The MEMM uses a conditional classifier with the previous tag as a history feature; the CRF learns transition and observation weights jointly.

## 0. Setup and dependencies

The CRF implementation uses [`sklearn-crfsuite`](https://sklearn-crfsuite.readthedocs.io/en/latest/tutorial.html). Install it once if it is not already available:

```python
%pip install sklearn-crfsuite
```

The notebook expects the data files under `Hindi-POS-Data/`. Run all cells from the `Assignment9` directory.

In [ ]:
from pathlib import Path
from collections import Counter
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

try:
    import sklearn_crfsuite
except ImportError as exc:
    raise ImportError("Install sklearn-crfsuite with: %pip install sklearn-crfsuite") from exc

DATA_DIR = Path("Hindi-POS-Data")
if not (DATA_DIR / "hindi-train-pos.txt").exists():
    DATA_DIR = Path("Assignment9/Hindi-POS-Data")
TRAIN_FILE = DATA_DIR / "hindi-train-pos.txt"
DEV_FILE = DATA_DIR / "hindi-dev-pos.txt"
TEST_FILE = DATA_DIR / "hindi-test-pos.txt"
OUTPUT_DIR = Path("pos_tagger_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Using data directory: {DATA_DIR.resolve()}")

## 1. Read the tagged Hindi data

Each non-empty line contains a word and tag separated by whitespace. Blank lines delimit sentences. Keeping sentence boundaries is important because both MEMM and CRF use sentence context.

In [ ]:
def read_tagged_sentences(path):
    sentences, current = [], []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                if current:
                    sentences.append(current)
                    current = []
                continue
            word, tag = line.rsplit(maxsplit=1)
            current.append((word, tag))
    if current:
        sentences.append(current)
    return sentences

train_sents = read_tagged_sentences(TRAIN_FILE)
dev_sents = read_tagged_sentences(DEV_FILE)
test_sents = read_tagged_sentences(TEST_FILE)
tags = sorted({tag for sentence in train_sents for _, tag in sentence})
print(pd.DataFrame({
    "split": ["train", "dev", "test"],
    "sentences": [len(train_sents), len(dev_sents), len(test_sents)],
    "tokens": [sum(map(len, train_sents)), sum(map(len, dev_sents)), sum(map(len, test_sents))],
}))
print(f"Training tag inventory: {len(tags)} tags")

## 2. Feature extraction

For every token, the feature function creates:

- `prefix-1` through `prefix-4`; missing lengths are represented by `NULL`.
- `suffix-1` through `suffix-4`; missing lengths are represented by `NULL`.
- Previous, current, and next word, with `BOS`/`EOS` sentence markers.
- `length=LESS` for words of length at most 4 and `length=MORE` otherwise.

The MEMM additionally receives `prev_tag`. During training this is the gold previous tag; during prediction it is the previously predicted tag.

In [ ]:
BOS, EOS, NULL = "<BOS>", "<EOS>", "NULL"

def token_features(words, i, previous_tag=None):
    word = words[i]
    features = {
        "word=prev": words[i - 1] if i else BOS,
        "word=cur": word,
        "word=next": words[i + 1] if i + 1 < len(words) else EOS,
        "length": "MORE" if len(word) > 4 else "LESS",
    }
    for n in range(1, 5):
        features[f"prefix-{n}"] = word[:n] if len(word) >= n else NULL
        features[f"suffix-{n}"] = word[-n:] if len(word) >= n else NULL
    if previous_tag is not None:
        features["prev_tag"] = previous_tag
    return features

def sentence_features(sentence, include_previous_tag=False, previous_tags=None):
    words = [word for word, _ in sentence]
    output = []
    for i in range(len(words)):
        previous = None
        if include_previous_tag:
            previous = previous_tags[i - 1] if i else BOS
        output.append(token_features(words, i, previous))
    return output

print(sentence_features(train_sents[0], include_previous_tag=True, previous_tags=[tag for _, tag in train_sents[0]])[0])

## 3. Maximum Entropy Markov Model (MEMM)

A MEMM estimates a locally normalized conditional distribution:

`P(tag_i | features_i, tag_{i-1})`

This implementation uses multinomial logistic regression as the maximum-entropy classifier. The previous tag is included as a feature, and decoding is greedy left-to-right. This is the same history-based conditional modeling principle used by the referenced NLTK maxent POS-tagger implementation.

In [ ]:
memm_rows, memm_labels = [], []
for sentence in train_sents:
    gold_tags = [tag for _, tag in sentence]
    memm_rows.extend(sentence_features(sentence, include_previous_tag=True, previous_tags=gold_tags))
    memm_labels.extend(gold_tags)

memm_vectorizer = DictVectorizer(sparse=True)
X_memm = memm_vectorizer.fit_transform(memm_rows)
memm = LogisticRegression(
    solver="saga", max_iter=100, n_jobs=-1, multi_class="multinomial", random_state=7
)
memm.fit(X_memm, memm_labels)

def predict_memm(sentence):
    words = [word for word, _ in sentence]
    predicted = []
    rows = []
    for i in range(len(words)):
        previous = predicted[i - 1] if i else BOS
        rows.append(token_features(words, i, previous))
    X = memm_vectorizer.transform(rows)
    return memm.predict(X).tolist()

print(f"MEMM training rows: {X_memm.shape[0]:,}")
print(f"MEMM feature columns: {X_memm.shape[1]:,}")

## 4. Conditional Random Field (CRF)

Unlike the MEMM, a CRF scores the complete tag sequence and normalizes over all possible sequences. `sklearn-crfsuite` performs sequence decoding and learns transition as well as feature weights. The CRF receives the required token features but does not need a manually supplied previous-tag feature because tag transitions are learned by the sequence model.

In [ ]:
X_crf = [sentence_features(sentence) for sentence in train_sents]
y_crf = [[tag for _, tag in sentence] for sentence in train_sents]

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)
crf.fit(X_crf, y_crf)

def predict_crf(sentence):
    return crf.predict([sentence_features(sentence)])[0]

print(f"CRF labels: {len(crf.classes_)}")

## 5. Evaluation on development and test sets

Accuracy is token accuracy. Macro-F1 gives every tag equal weight, while weighted-F1 weights each tag by its support. Per-tag F1 is also saved for the required comparison.

In [ ]:
def evaluate(name, predictor, sentences, split):
    y_true, y_pred = [], []
    for sentence in sentences:
        gold = [tag for _, tag in sentence]
        pred = predictor(sentence)
        if len(gold) != len(pred):
            raise ValueError(f"{name} returned the wrong number of tags")
        y_true.extend(gold)
        y_pred.extend(pred)
    report = classification_report(y_true, y_pred, labels=tags, output_dict=True, zero_division=0)
    summary = {
        "split": split, "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }
    per_tag = [{"split": split, "model": name, "tag": tag, "precision": report[tag]["precision"],
                "recall": report[tag]["recall"], "f1": report[tag]["f1-score"], "support": report[tag]["support"]}
               for tag in tags]
    return summary, per_tag

summary_rows, per_tag_rows = [], []
for split, sentences in [("dev", dev_sents), ("test", test_sents)]:
    for name, predictor in [("MEMM", predict_memm), ("CRF", predict_crf)]:
        summary, per_tag = evaluate(name, predictor, sentences, split)
        summary_rows.append(summary)
        per_tag_rows.extend(per_tag)

summary_results = pd.DataFrame(summary_rows)
per_tag_results = pd.DataFrame(per_tag_rows)
display(summary_results.round(4))
per_tag_results.to_csv(OUTPUT_DIR / "memm_crf_per_tag_metrics.csv", index=False)
summary_results.to_csv(OUTPUT_DIR / "memm_crf_summary_metrics.csv", index=False)

## 6. Comparison with Assignment 7

Assignment 7's generated CSV files are loaded when available. This avoids silently re-running a different implementation and preserves the original HMM and Naive Bayes measurements. If those files are absent, the notebook still reports MEMM and CRF results and explains which comparison data is missing.

In [ ]:
previous_summary_paths = [
    Path("../Assignment7/pos_tagger_outputs/summary_metrics.csv"),
    Path("Assignment7/pos_tagger_outputs/summary_metrics.csv"),
]
previous_per_tag_paths = [
    Path("../Assignment7/pos_tagger_outputs/per_tag_metrics.csv"),
    Path("Assignment7/pos_tagger_outputs/per_tag_metrics.csv"),
]
summary_path = next((path for path in previous_summary_paths if path.exists()), None)
per_tag_path = next((path for path in previous_per_tag_paths if path.exists()), None)

comparison = summary_results.copy()
comparison_per_tag = per_tag_results.copy()
if summary_path is not None:
    old_summary = pd.read_csv(summary_path)
    keep = ["split", "model", "accuracy", "macro_f1", "weighted_f1"]
    comparison = pd.concat([old_summary[keep], comparison], ignore_index=True)
    if per_tag_path is not None:
        old_per_tag = pd.read_csv(per_tag_path)
        comparison_per_tag = pd.concat([old_per_tag, comparison_per_tag], ignore_index=True)
    print(f"Loaded Assignment 7 comparison results from {summary_path}")
else:
    print("Assignment 7 summary CSV not found; showing MEMM and CRF only.")

display(comparison.pivot(index="model", columns="split", values=["accuracy", "macro_f1", "weighted_f1"]).round(4))
comparison.to_csv(OUTPUT_DIR / "all_model_summary_metrics.csv", index=False)
comparison_per_tag.to_csv(OUTPUT_DIR / "all_model_per_tag_metrics.csv", index=False)

In [ ]:
for metric in ["macro_f1", "weighted_f1"]:
    pivot = comparison.pivot(index="model", columns="split", values=metric)
    pivot.plot(kind="bar", figsize=(10, 5), ylim=(0, 1), title=metric.replace("_", " ").title())
    plt.ylabel(metric.replace("_", " ").title())
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

test_f1 = comparison_per_tag[comparison_per_tag["split"] == "test"].pivot(
    index="tag", columns="model", values="f1"
)
test_f1.plot(kind="bar", figsize=(14, 6), ylim=(0, 1), title="Test-set F1 by POS tag")
plt.ylabel("F1 score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 7. Interpretation

- MEMM makes locally normalized decisions and uses its previous prediction during decoding, so early errors can influence later tags.
- CRF performs global sequence scoring and learns tag-transition patterns jointly with lexical and context features.
- Macro-F1 is especially useful for Hindi POS tagging because rare tags can be hidden by high-frequency tags in accuracy and weighted-F1.
- The final tables and plots provide the required comparison of MEMM, CRF, Naive Bayes, and HMM. Exact numbers depend on the optimizer, regularization, and the Assignment 7 result files loaded above.

Generated files:

- `pos_tagger_outputs/memm_crf_summary_metrics.csv`
- `pos_tagger_outputs/memm_crf_per_tag_metrics.csv`
- `pos_tagger_outputs/all_model_summary_metrics.csv`
- `pos_tagger_outputs/all_model_per_tag_metrics.csv`